# Silver: limpeza e padronizacao

Este notebook le as tabelas Bronze, remove registros sem chaves, converte datas, valores numericos e notas de avaliacao, e grava os dados tratados em `workspace.olist_silver`.

A nota `review_score` aceita somente valores de 1 a 5. Valores invalidos sao ignorados com `try_cast`.

**Atencao especial para as hipoteses deste MVP:**
- `products`: peso e dimensoes fisicas (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`) sao convertidos para `double`. Valores <= 0 sao removidos (fisicamente impossiveis); nulos sao mantidos, pois a ausencia de dado nao invalida o produto para outras analises — a exclusao explicita acontece na query da Pergunta 1.
- `order_items`: `seller_id` passa a ser obrigatorio (NOT NULL), pois e a chave central da Pergunta 2.
- `payments`: `payment_installments` e `payment_value` sao tipados, para uso na fato_vendas.

**Ordem de execucao:** execute o Bronze antes deste notebook e o Gold depois dele.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver — limpeza, padronização e métricas de qualidade

# COMMAND ----------

from pyspark.sql import Window, functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "olist_bronze"
SILVER_SCHEMA = "olist_silver"
BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG}.{SILVER_SCHEMA}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def read_csv(file_name):
    table_name = {
        "olist_orders_dataset.csv": "orders",
        "olist_customers_dataset.csv": "customers",
        "olist_products_dataset.csv": "products",
        "olist_order_items_dataset.csv": "order_items",
        "olist_order_payments_dataset.csv": "payments",
        "olist_order_reviews_dataset.csv": "reviews",
        "olist_sellers_dataset.csv": "sellers",
        "olist_geolocation_dataset.csv": "geolocation",
    }[file_name]
    return spark.table(f"{BRONZE}.{table_name}")


def first_by_key(dataframe, keys):
    return dataframe.withColumn(
        "_row_number", F.row_number().over(Window.partitionBy(*keys).orderBy(F.lit(1)))
    ).filter("_row_number = 1").drop("_row_number")


def save(dataframe, table_name):
    dataframe.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.{table_name}")


# COMMAND ----------

orders_raw = read_csv("olist_orders_dataset.csv")
orders_with_keys = orders_raw.filter("order_id IS NOT NULL AND customer_id IS NOT NULL")
orders = first_by_key(orders_with_keys, ["order_id"])
for column in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    orders = orders.withColumn(column, F.to_timestamp(column))
save(orders, "orders")

customers_raw = read_csv("olist_customers_dataset.csv")
customers = first_by_key(customers_raw.filter("customer_id IS NOT NULL"), ["customer_id"])
save(customers, "customers")

items_raw = read_csv("olist_order_items_dataset.csv")
items_with_keys = items_raw.filter(
    "order_id IS NOT NULL AND product_id IS NOT NULL AND seller_id IS NOT NULL"
)
items = first_by_key(
    items_with_keys, ["order_id", "order_item_id"]
).withColumn("order_item_id", F.col("order_item_id").cast("int")).withColumn(
    "price", F.col("price").cast("decimal(12,2)")
).withColumn("freight_value", F.col("freight_value").cast("decimal(12,2)"))
save(items, "order_items")

# Peso e dimensoes fisicas viram double (chegam como string do Bronze) para
# alimentar a razao frete/kg e o volume da Pergunta 1.
campos_fisicos = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]
products_raw = read_csv("olist_products_dataset.csv")
products_with_id = products_raw.filter("product_id IS NOT NULL")
products_typed = first_by_key(products_with_id, ["product_id"]).withColumn(
    "product_category_name", F.coalesce("product_category_name", F.lit("sem_categoria"))
)
for column in campos_fisicos:
    products_typed = products_typed.withColumn(column, F.col(column).cast("double"))

# Mantem nulos (ausencia de informacao); remove apenas valores <= 0 (fisicamente impossiveis).
products = products_typed
for column in campos_fisicos:
    products = products.filter((F.col(column).isNull()) | (F.col(column) > 0))
save(products, "products")

# COMMAND ----------

reviews_raw = read_csv("olist_order_reviews_dataset.csv")
reviews = (
    first_by_key(reviews_raw.filter("order_id IS NOT NULL"), ["order_id"])
    .withColumn("review_score", F.expr("try_cast(review_score AS INT)"))
    .filter("review_score BETWEEN 1 AND 5")
)
save(reviews, "reviews")

payments_raw = read_csv("olist_order_payments_dataset.csv")
payments = (
    payments_raw.filter("order_id IS NOT NULL")
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .withColumn("payment_value", F.col("payment_value").cast("decimal(12,2)"))
)
payments = first_by_key(payments, ["order_id", "payment_sequential"])
save(payments, "payments")

# COMMAND ----------

quality_metrics = [
    ("orders", "completude", "order_id_nulo", orders_raw.filter("order_id IS NULL").count()),
    ("orders", "unicidade", "order_id_duplicado", orders_with_keys.count() - orders.count()),
    ("orders", "acuracia", "entrega_acima_500_dias", orders.filter(
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp") > 500
    ).count()),
    ("reviews", "completude", "review_score_nulo", reviews_raw.filter("review_score IS NULL").count()),
    ("items", "outliers", "frete_acima_p99", items.filter(
        F.col("freight_value") > items.approxQuantile("freight_value", [0.99], 0.01)[0]
    ).count()),
    ("items", "completude", "seller_id_nulo", items_raw.filter("seller_id IS NULL").count()),
    ("products", "completude", "peso_ou_dimensao_nula", products_typed.filter(
        " OR ".join(f"{coluna} IS NULL" for coluna in campos_fisicos)
    ).count()),
    ("products", "acuracia", "peso_ou_dimensao_invalida_removida", products_typed.count() - products.count()),
]
spark.createDataFrame(
    quality_metrics, "table_name string, dimension string, metric string, metric_value long"
).withColumn("measured_at", F.current_timestamp()).write.format("delta").mode(
    "overwrite"
).saveAsTable(f"{SILVER}.data_quality_metrics")
